In [1]:
import pandas as pd

RAW = "../data/raw/off_products.csv.gz"

cols = pd.read_csv(RAW, sep="\t", nrows=0).columns
print(f"Columnas totales: {len(cols)}")

interesantes = [c for c in cols if any(k in c for k in ["nova", "_100g", "pnns", "additives", "ingredients_text", "countries"])]
interesantes

Columnas totales: 211


['countries',
 'countries_tags',
 'countries_en',
 'ingredients_text',
 'additives_n',
 'additives',
 'additives_tags',
 'additives_en',
 'nova_group',
 'pnns_groups_1',
 'pnns_groups_2',
 'energy-kj_100g',
 'energy-kcal_100g',
 'energy_100g',
 'energy-from-fat_100g',
 'fat_100g',
 'saturated-fat_100g',
 'butyric-acid_100g',
 'caproic-acid_100g',
 'caprylic-acid_100g',
 'capric-acid_100g',
 'lauric-acid_100g',
 'myristic-acid_100g',
 'palmitic-acid_100g',
 'stearic-acid_100g',
 'arachidic-acid_100g',
 'behenic-acid_100g',
 'lignoceric-acid_100g',
 'cerotic-acid_100g',
 'montanic-acid_100g',
 'melissic-acid_100g',
 'unsaturated-fat_100g',
 'monounsaturated-fat_100g',
 'omega-9-fat_100g',
 'polyunsaturated-fat_100g',
 'omega-3-fat_100g',
 'omega-6-fat_100g',
 'alpha-linolenic-acid_100g',
 'eicosapentaenoic-acid_100g',
 'docosahexaenoic-acid_100g',
 'linoleic-acid_100g',
 'arachidonic-acid_100g',
 'gamma-linolenic-acid_100g',
 'dihomo-gamma-linolenic-acid_100g',
 'oleic-acid_100g',
 'elai

In [2]:
USECOLS = [
    "code", "product_name", "brands",
    "countries_tags", "categories_tags", "pnns_groups_1", "pnns_groups_2",
    "ingredients_text", "additives_n", "additives_tags",
    "nova_group",
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g",
    "carbohydrates_100g", "sugars_100g", "fiber_100g",
    "proteins_100g", "salt_100g", "sodium_100g",
    "fruits-vegetables-legumes_100g",
]

faltan = [c for c in USECOLS if c not in cols]
print("Columnas que no existen:", faltan)

trozos = []
total = 0
for chunk in pd.read_csv(
        RAW, sep="\t", usecols=USECOLS,
        chunksize=200_000,
        low_memory=False,
        on_bad_lines="skip",
):
    total += len(chunk)
    es = chunk[chunk["countries_tags"].str.contains("en:spain", na=False)]
    trozos.append(es)

df = pd.concat(trozos, ignore_index=True)
print(f"Filas totales: {total:,} -> España: {len(df):,}")

df.to_csv("../data/interim/off_spain.csv.gz", index=False)

Columnas que no existen: []
Filas totales: 4,906,687 -> España: 362,954


In [3]:
print(f"Productos en España:        {len(df):,}")
print(f"Con NOVA:                   {df['nova_group'].notna().sum():,}")
print(f"Con NOVA e ingredientes:    {(df['nova_group'].notna() & df['ingredients_text'].notna()).sum():,}")

df["nova_group"].value_counts(dropna=False).sort_index()

Productos en España:        362,954
Con NOVA:                   54,403
Con NOVA e ingredientes:    46,561


nova_group
1.0      6996
2.0      8038
3.0     10390
4.0     28979
NaN    308551
Name: count, dtype: int64